# Comprehensions

This module covers Python's comprehension forms: list, set, dict, and generator expressions. Readers are assumed to be comfortable with Python syntax, with `for` loops and the basic container types, and with at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on how comprehensions read, when they fit tm1py work, and where the traps lie.

A comprehension rolls a `for` loop, an optional filter, and a transformation into a single expression that produces a container or an iterator. The TM1 cellset, which arrives from `execute_view_values` as a `dict[tuple[str, ...], float]` keyed by element names, is a natural fit for every shape comprehensions take, and is the running example throughout the module.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

https://www.datacamp.com/tutorial/python-list-comprehension
https://www.datacamp.com/tutorial/python-dictionary-comprehension

---

## Topic list

1. Comprehensions in Python
2. List comprehensions
3. Filtering with `if`
4. Conditional expressions on the output
5. Nested loops
6. Set comprehensions
7. Dict comprehensions
8. Generator expressions
9. The walrus operator inside comprehensions
10. Comprehensions vs `map` and `filter`
11. Scope: what leaks and what does not
12. Real world design principles
13. Common mistakes

---

## 1. Comprehensions in Python

Python has four comprehension forms, all built from the same skeleton: a transformation expression, one or more `for` clauses, and zero or more `if` filters. They differ only in the brackets that surround them:

In [ ]:
[expr for x in src]            # list comprehension
{expr for x in src}            # set comprehension
{key: val for x in src}        # dict comprehension
(expr for x in src)            # generator expression

Each one reads top down: produce `expr` for every `x` drawn from `src` that passes the optional filter. The result is a list, a set, a dict, or a lazy iterator respectively. Older code often builds the same shape with an explicit accumulator and a `for` loop; the comprehension is shorter, faster (the loop runs in C inside CPython's compiled bytecode), and signals intent: this is a transformation of an iterable, not an arbitrary loop that happens to append.

The forms exist because Python takes container building seriously enough to give each container its own literal syntax. Once a reader recognizes the four bracket pairs, the rest is just the body.

## 2. List comprehensions

The list form is the most common. Given a tm1py cellset returned from `execute_view_values`, where each key is a tuple of element names and each value is the cell value, a list comprehension extracts the revenue cells for one region:

In [ ]:
cells: dict[tuple[str, ...], float] = tm1.cells.execute_view_values(
    cube_name="Sales Plan", view_name="Plan Input"
)
# cells: { ("2026", "Jan", "Europe", "Phones", "Plan", "Revenue"): 120_000.0,
#          ("2026", "Feb", "Europe", "Phones", "Plan", "Revenue"): 135_000.0, ... }

europe_revenue: list[float] = [
    value
    for (year, period, region, product, version, measure), value in cells.items()
    if region == "Europe" and measure == "Revenue"
]
# europe_revenue: [120_000.0, 135_000.0, 142_500.0, ...]

The transformation is `value`, the source is `cells.items()`, and the filter is the `if` clause. The result is a list of floats in the iteration order of the dict, which since Python 3.7 is insertion order. The same loop written with `append`:

In [ ]:
europe_revenue = []
for (year, period, region, product, version, measure), value in cells.items():
    if region == "Europe" and measure == "Revenue":
        europe_revenue.append(value)

Both produce the same list, but the comprehension form is what most Python code uses, and it is what type checkers and most readers expect for this shape of operation. The accumulator form is reserved for loops with side effects or with logic that does not fit the single expression skeleton.

## 3. Filtering with `if`

A trailing `if` clause filters the source. Multiple `if` clauses work as a logical AND, evaluated left to right, which lets the cheaper checks run first:

In [ ]:
plan_keys: list[tuple[str, ...]] = [
    key
    for key in cells
    if key[4] == "Plan"                       # cheap tuple index check first
    if key[2] in {"Europe", "Americas"}       # set membership next
    if cells[key] > 0                         # value lookup last
]

The filter does not introduce a separate scope; it sees every variable bound by the `for` clause. Combining filters with `and` produces the same result as chaining `if` clauses, and CPython compiles them the same way; the chained form is preferred when the filters are conceptually independent, the joined form when they express a single condition.

A comprehension whose filter excludes most input is often a candidate for a generator expression instead (see Topic 8); building the full intermediate list only to throw most of it away is wasted memory.

## 4. Conditional expressions on the output

A conditional expression (`a if cond else b`) in the transformation slot is different from an `if` filter. The filter decides whether the row is included; the conditional expression decides what value the row takes when it is included:

In [ ]:
flagged: list[tuple[tuple[str, ...], str]] = [
    (key, "negative" if value < 0 else "ok")
    for key, value in cells.items()
    if key[5] == "Revenue"
]
# flagged: [(("2026", "Jan", "Europe", "Phones", "Plan", "Revenue"), "ok"),
#           (("2026", "Feb", "Asia",   "Phones", "Plan", "Revenue"), "negative"),
#           ...]

The two `if`s look similar but sit in different slots. The leading `if` is part of the expression and must have an `else`; the trailing `if` is the filter and stands alone. A common confusion is trying to filter with a leading `if`:

In [ ]:
# Wrong: leading if without else is a SyntaxError
[value if value > 0 for value in numbers]

# Correct: trailing if is the filter
[value for value in numbers if value > 0]

If both shaping and filtering are needed, both `if`s appear in the same comprehension, in their respective slots.

## 5. Nested loops

A comprehension can have several `for` clauses. They nest left to right, the same order as if they were written as nested `for` blocks. A common tm1py case is producing the cross product of two dimensions for an MDX query:

In [ ]:
regions: list[str] = ["Europe", "Americas", "Asia"]
products: list[str] = ["Phones", "Tablets", "Laptops"]

tuples: list[str] = [
    f"([Region].[{region}], [Product].[{product}])"
    for region in regions
    for product in products
]
# tuples[0:3]: ["([Region].[Europe], [Product].[Phones])",
#               "([Region].[Europe], [Product].[Tablets])",
#               "([Region].[Europe], [Product].[Laptops])"]

Inner clauses can reference variables from outer clauses, which is how filters that depend on the outer iteration are written:

In [ ]:
sold_in: dict[str, list[str]] = {
    "Europe":   ["Phones", "Tablets"],
    "Americas": ["Phones", "Tablets", "Laptops"],
    "Asia":     ["Phones"],
}

valid_tuples: list[str] = [
    f"([Region].[{region}], [Product].[{product}])"
    for region in regions
    for product in sold_in[region]
]

Two `for` clauses are usually fine to read; three is the practical limit. Beyond that, an explicit nested loop or a helper function is clearer than a tall comprehension.

## 6. Set comprehensions

Set comprehensions use braces and produce a `set`, removing duplicates as they go. The natural use is collecting unique element names from a cellset:

In [ ]:
regions_present: set[str] = {key[2] for key in cells}
# regions_present: {"Europe", "Americas", "Asia"}

months_with_revenue: set[str] = {
    key[1]
    for key, value in cells.items()
    if key[5] == "Revenue" and value > 0
}
# months_with_revenue: {"Jan", "Feb", "Mar", "Apr", ...}

Because the result is a set, iteration order is not preserved. Reach for a set comprehension when the answer is "which distinct values appear" rather than "what is each value"; the deduplication is the point. A list comprehension followed by `set(...)` produces the same answer but builds the list first; the set form skips that intermediate.

## 7. Dict comprehensions

Dict comprehensions produce a `dict` and are the right shape for any "key by X, value by Y" transformation. A frequent tm1py use is rekeying a cellset by a slice of the original tuple, for instance dropping the version and measure to get a (year, period, region, product) keyed view of plan revenue:

In [ ]:
revenue_only: dict[tuple[str, str, str, str], float] = {
    (year, period, region, product): value
    for (year, period, region, product, version, measure), value in cells.items()
    if version == "Plan" and measure == "Revenue"
}
# revenue_only: { ("2026", "Jan", "Europe", "Phones"): 120_000.0,
#                 ("2026", "Feb", "Europe", "Phones"): 135_000.0, ... }

Inverting a dict is a one liner:

In [ ]:
element_index: dict[str, int] = {"Jan": 0, "Feb": 1, "Mar": 2, "Apr": 3}
index_to_element: dict[int, str] = {i: name for name, i in element_index.items()}
# index_to_element: {0: "Jan", 1: "Feb", 2: "Mar", 3: "Apr"}

Two notes. First, if the same key appears twice the later value wins silently; this is a common source of bugs when the keying expression is not actually unique (see the aggregation pitfall in Topic 13). Second, the comprehension is one expression and cannot mutate the dict it is building, which is part of what lets CPython compile the form efficiently.

## 8. Generator expressions

Replacing the brackets with parentheses produces a generator expression: a lazy iterator that yields values on demand instead of materializing a container.

In [ ]:
total_revenue: float = sum(
    value
    for key, value in cells.items()
    if key[5] == "Revenue"
)
# total_revenue: 4_872_500.0

The generator is consumed by `sum`; no intermediate list is built. For a cellset of a few thousand entries the difference is invisible, but `execute_view_values` on a real planning cube can return millions of cells, where the list form holds every value in memory and the generator does not.

When a generator expression is the sole argument to a function, the surrounding parentheses can be omitted:

In [ ]:
total_revenue = sum(value for key, value in cells.items() if key[5] == "Revenue")

This is the only place the parentheses can be dropped. Anywhere else, the parentheses are required. A generator is single use: once iterated, it is exhausted, and a second pass produces nothing. If two passes are needed, materialize with `list(...)` or use a list comprehension to begin with.

## 9. The walrus operator inside comprehensions

The walrus operator `:=` (Python 3.8+) binds a value to a name as part of an expression. Inside a comprehension it is useful when the same computed value is needed in both the filter and the result, and computing it twice would be wasteful:

In [ ]:
def lookup_target(region: str, product: str) -> float | None:
    # an expensive lookup, perhaps a tm1py call or a cached function
    ...

results: list[tuple[str, str, float]] = [
    (region, product, target)
    for region in regions
    for product in products
    if (target := lookup_target(region, product)) is not None
]

Without the walrus, the choice is to call `lookup_target` twice (once in the filter, once in the result) or to fall back to an explicit loop. The walrus binds inside the comprehension's implicit function scope, so `target` is not visible after the comprehension ends. Reach for it only when it removes a duplicated computation; using it for style alone makes the comprehension harder to read.

## 10. Comprehensions vs `map` and `filter`

`map(fn, iterable)` and `filter(fn, iterable)` are the older functional forms and return iterators, not lists, in Python 3. Anything they do, a comprehension or generator expression also does:

In [ ]:
# map and filter form
keys = list(filter(lambda k: k[5] == "Revenue", cells))
values = list(map(lambda v: v * 1.1, cells.values()))

# comprehension form
keys = [k for k in cells if k[5] == "Revenue"]
values = [v * 1.1 for v in cells.values()]

The comprehension form is preferred in modern Python for two reasons. The body of a comprehension is a normal expression with full access to the surrounding scope, while `map` and `filter` need a callable, which usually means a `lambda`. And comprehensions express the transformation and the filter together, while `map` or `filter` chains require nesting or repeated wrapping. `map` is still idiomatic when the callable is already named (`map(str.upper, names)`); a comprehension is the better choice whenever the body is anything more than a single function call.

## 11. Scope: what leaks and what does not

In Python 3, the loop variable inside a comprehension is local to the comprehension. The following raises `NameError`:

In [ ]:
[x * 2 for x in range(10)]
print(x)   # NameError: name 'x' is not defined

This differs from a plain `for` loop, where `x` would persist after the loop. The change was made in the Python 2 to 3 transition because comprehension variables leaking into the enclosing namespace was a frequent source of bugs (a comprehension named `i` would clobber an outer `i` used later). Generator expressions have always been scoped this way.

Class body comprehensions are an edge case worth knowing about: a comprehension inside a class body cannot see other class level names directly, because the comprehension runs in its own implicit function scope:

In [ ]:
class CubeMap:
    cubes = ["Sales Plan", "General Ledger", "Headcount"]
    upper_cubes = [name.upper() for name in cubes]              # works
    indexed = {name: i for i, name in enumerate(cubes)}         # works
    # but referring to a sibling class attribute inside the body would fail

For ordinary module level and function level use, the rule is simple: the comprehension's loop variable is private to the comprehension.

## 12. Real world design principles

**Reach for a comprehension when the loop's purpose is to build a container.** If the code creates an empty list, set, or dict and then loops to fill it, the comprehension form is shorter and clearer. If the loop has side effects (logging, writing cells back to TM1, sending HTTP requests), keep it as an explicit `for` block.

**Prefer the generator form for large tm1py cellsets.** A planning cube view that returns millions of cells should be summed, filtered, or transformed through a generator, not a list. The generator form is one character different and avoids holding the whole result in memory.

**Stop at two `for` clauses.** A nested comprehension with three or more `for`s reads as a puzzle. Either flatten with `itertools.product`, extract a helper function, or use plain nested loops. The goal of the comprehension is clarity, not density.

**Pull a long body out into a function.** A comprehension whose body spans three lines is a function in disguise. Name the transformation, then map it.

**Use dict comprehensions to reshape cellsets, not to rebuild them.** When `execute_view_values` already returns a dict, a comprehension to drop unused dimensions or to rekey by a different slice is the natural shape. Building a fresh dict cell by cell with a comprehension is rare; it usually means the wrong tm1py call was used.

**Treat the comprehension as one expression for review.** If a colleague cannot read a comprehension at a glance, the comprehension is too clever. Splitting the transformation across a named helper and a thin comprehension is almost always more readable than packing everything into one expression.

## 13. Common mistakes

**Confusing the filter `if` with the conditional expression.**

In [ ]:
# Wrong: leading if without else is a SyntaxError
[value if value > 0 for value in numbers]

# Correct: filter goes after the for
[value for value in numbers if value > 0]

**Materializing a generator just to count or sum it.**

In [ ]:
# Wrong
total = sum([value for key, value in cells.items() if key[5] == "Revenue"])

# Correct
total = sum(value for key, value in cells.items() if key[5] == "Revenue")

**Iterating a generator twice.**

In [ ]:
# Wrong
gen = (k for k in cells if k[5] == "Revenue")
count = sum(1 for _ in gen)
first = next(iter(gen))     # generator is already exhausted, raises StopIteration

# Correct
keys = [k for k in cells if k[5] == "Revenue"]
count = len(keys)
first = keys[0]

**Silent key collisions in dict comprehensions.**

In [ ]:
# Wrong: keying by region drops every period down to one value per region
by_region = {key[2]: value for key, value in cells.items() if key[5] == "Revenue"}
# only the last (region, value) pair survives for each region

# Correct: aggregate explicitly
from collections import defaultdict
by_region: dict[str, float] = defaultdict(float)
for key, value in cells.items():
    if key[5] == "Revenue":
        by_region[key[2]] += value

**Using a comprehension purely for its side effect.**

In [ ]:
# Wrong: throws the resulting list away
[tm1.cells.write_value(value, "Sales Plan", key) for key, value in updates.items()]

# Correct: a plain loop reads as what it is
for key, value in updates.items():
    tm1.cells.write_value(value, "Sales Plan", key)

**Mutating the source while iterating.**

In [ ]:
# Wrong: pop mutates cells during iteration
filtered = [k for k in cells if cells.pop(k, 0) > 0]

# Correct
filtered = [k for k, v in cells.items() if v > 0]